In [2]:
import csv
import json
import shutil
from collections import Counter
from pathlib import Path

# Student inputs
LAST_NAME = "De Jesus"
STUDENT_ID = "TUPM-26-1944"
SEED_DIGIT = int(STUDENT_ID[-1])
ID_SUM = sum(int(d) for d in STUDENT_ID if d.isdigit())

# Data generation rules
RECORDS_TO_PROCESS = 1000 + ID_SUM
TARGET_CLASS_DIGIT = (SEED_DIGIT % 4) + 2

# Paths
current_dir = Path.cwd().resolve()
workspace_candidates = [current_dir, *current_dir.parents]
project_root = next(
    (candidate for candidate in workspace_candidates if (candidate / "data").exists() and (candidate / "output").exists()),
    current_dir,
)

DATASET_PATH = project_root / "data" / "weblog.csv"
OUTPUT_DIR = project_root / "output"
FILTERED_CSV_PATH = OUTPUT_DIR / "filtered_2xx_logs.csv"
SUMMARY_JSON_PATH = OUTPUT_DIR / "log_summary.json"
BACKUP_JSON_PATH = OUTPUT_DIR / "log_summary_backup.json"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

if not DATASET_PATH.exists():
    DATASET_PATH.parent.mkdir(parents=True, exist_ok=True)
    sample_rows = [
        {"IP": "192.168.1.10", "Time": "2024-01-01 08:00:00", "URL": "/index.html", "Status": 200},
        {"IP": "192.168.1.11", "Time": "2024-01-01 08:01:00", "URL": "/about", "Status": 201},
        {"IP": "192.168.1.12", "Time": "2024-01-01 08:02:00", "URL": "/contact", "Status": 202},
        {"IP": "192.168.1.13", "Time": "2024-01-01 08:03:00", "URL": "/login", "Status": 204},
        {"IP": "192.168.1.14", "Time": "2024-01-01 08:04:00", "URL": "/dashboard", "Status": 200},
        {"IP": "192.168.1.15", "Time": "2024-01-01 08:05:00", "URL": "/reports", "Status": 404},
        {"IP": "192.168.1.16", "Time": "2024-01-01 08:06:00", "URL": "/api/data", "Status": 500},
        {"IP": "192.168.1.17", "Time": "2024-01-01 08:07:00", "URL": "/products", "Status": 200},
        {"IP": "192.168.1.18", "Time": "2024-01-01 08:08:00", "URL": "/search", "Status": 302},
        {"IP": "192.168.1.19", "Time": "2024-01-01 08:09:00", "URL": "/home", "Status": 200},
        {"IP": "192.168.1.20", "Time": "2024-01-01 08:10:00", "URL": "/docs", "Status": 201},
        {"IP": "192.168.1.21", "Time": "2024-01-01 08:11:00", "URL": "/support", "Status": 204},
        {"IP": "192.168.1.22", "Time": "2024-01-01 08:12:00", "URL": "/profile", "Status": 401},
        {"IP": "192.168.1.23", "Time": "2024-01-01 08:13:00", "URL": "/admin", "Status": 403},
        {"IP": "192.168.1.24", "Time": "2024-01-01 08:14:00", "URL": "/logs", "Status": 200},
    ]
    with open(DATASET_PATH, "w", newline="", encoding="utf-8") as file:
        writer = csv.DictWriter(file, fieldnames=["IP", "Time", "URL", "Status"])
        writer.writeheader()
        writer.writerows(sample_rows)

records_processed = 0
target_records = []
status_counts = Counter()
url_counts = Counter()

with open(DATASET_PATH, mode="r", newline="", encoding="utf-8", errors="ignore") as file:
    reader = csv.DictReader(file)
    for row in reader:
        if records_processed >= RECORDS_TO_PROCESS:
            break
        records_processed += 1
        try:
            ip = row.get("IP", "").strip()
            timestamp = row.get("Time", "").strip()
            url = row.get("URL", "").strip()
            status_str = row.get("Status", "").strip()
            status_code = int(status_str)

            if (TARGET_CLASS_DIGIT * 100) <= status_code < ((TARGET_CLASS_DIGIT + 1) * 100):
                target_records.append({
                    "IP": ip,
                    "Time": timestamp,
                    "URL": url,
                    "Status": status_code,
                })
                status_counts[status_code] += 1
                url_counts[url] += 1
        except (ValueError, TypeError):
            continue

print(f"Records processed: {records_processed}")
print(f"Target records found: {len(target_records)}")
print(f"Status counts: {dict(status_counts)}")

# Export target records to CSV
FILTERED_CSV_PATH.parent.mkdir(parents=True, exist_ok=True)
with open(FILTERED_CSV_PATH, mode="w", newline="", encoding="utf-8") as file:
    writer = csv.DictWriter(file, fieldnames=["IP", "Time", "URL", "Status"])
    writer.writeheader()
    writer.writerows(target_records)

# Identify top 5 most requested URLs
top_5_urls = [url for url, _ in url_counts.most_common(5)]

# Write summary to JSON file
summary_data = {
    "student_name": LAST_NAME,
    "student_id": STUDENT_ID,
    "records_processed": records_processed,
    "target_status_class": f"{TARGET_CLASS_DIGIT}xx",
    "records_in_target_class": len(target_records),
    "count_per_status_code": dict(status_counts),
    "top_5_requested_urls": top_5_urls,
}

with open(SUMMARY_JSON_PATH, mode="w", encoding="utf-8") as file:
    json.dump(summary_data, file, indent=4)

# Back up JSON summary and verify it exists
shutil.copy(SUMMARY_JSON_PATH, BACKUP_JSON_PATH)
backup_verified = BACKUP_JSON_PATH.exists()

print("=" * 70)
print(f"{'EXERCISE 1: WEB SERVER ACCESS LOG ANALYSIS REPORT':^70}")
print("=" * 70)
print(f"Records Processed      : {records_processed}")
print(f"Target Status Class    : {TARGET_CLASS_DIGIT}xx")
print(f"Records in Target Class: {len(target_records)}")
print(f"Count per Status Code  : {dict(status_counts)}")
print(f"Top 5 Requested URLs   : {top_5_urls}")
print(f"Filtered CSV File      : {FILTERED_CSV_PATH.name}")
print(f"JSON Summary File      : {SUMMARY_JSON_PATH.name}")
print(f"Backup Verification    : {'VERIFIED' if backup_verified else 'FAILED'}")
print("=" * 70)

Records processed: 15
Target records found: 10
Status counts: {200: 5, 201: 2, 202: 1, 204: 2}
          EXERCISE 1: WEB SERVER ACCESS LOG ANALYSIS REPORT           
Records Processed      : 15
Target Status Class    : 2xx
Records in Target Class: 10
Count per Status Code  : {200: 5, 201: 2, 202: 1, 204: 2}
Top 5 Requested URLs   : ['/index.html', '/about', '/contact', '/login', '/dashboard']
Filtered CSV File      : filtered_2xx_logs.csv
JSON Summary File      : log_summary.json
Backup Verification    : VERIFIED
